# 🌿 02 Training — Sistem LSTM Bonsai

| Item       | Detail                                      |
|------------|---------------------------------------------|
| **File**   | `notebooks/02_training.ipynb`                |
| **Tujuan** | Bangun arsitektur LSTM, latih model dengan data training, simpan model & histori. |
| **Input**  | `artifacts/data_train.npz`, `artifacts/label_info.json` |
| **Output** | `artifacts/model_bonsai_lstm.keras`, `artifacts/training_history.csv` |
| **Urutan** | Jalankan setelah: `01_preprocessing.ipynb` |

In [1]:
# ── VERIFIKASI LINGKUNGAN ──────────────────────────────────────────────
import sys, os

assert ".venv" in sys.executable or "venv" in sys.executable, (
    "⛔ Kernel bukan dari .venv!\n"
    "Ganti kernel ke: Python (bonsai-lstm)\n"
    f"Kernel saat ini: {sys.executable}"
)
print("✅ Kernel  :", sys.executable)
print("✅ Python  :", sys.version)
# ──────────────────────────────────────────────────────────────────────

✅ Kernel  : D:\bonsai-lstm\.venv\Scripts\python.exe
✅ Python  : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
# ── IMPORT LIBRARY, KONSTANTA, SEED ───────────────────────────────────
import random, os
import warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"[SEED] Global random seed = {SEED}")

ARTIFACTS_DIR  = "../artifacts"
MODEL_PATH     = f"{ARTIFACTS_DIR}/model_bonsai_lstm.keras"
HISTORY_PATH   = f"{ARTIFACTS_DIR}/training_history.csv"

EPOCHS         = 100
BATCH_SIZE     = 32
LEARNING_RATE  = 0.0001
VAL_SPLIT      = 0.20
PATIENCE       = 25

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("✅ Konfigurasi training siap.")

[SEED] Global random seed = 42
✅ Konfigurasi training siap.


In [3]:
# ── RULE-NB-06: Validasi Artefak ───────────────────────────────────────
REQUIRED_ARTIFACTS = [
    f"{ARTIFACTS_DIR}/data_train.npz",
    f"{ARTIFACTS_DIR}/label_info.json",
]
missing = [f for f in REQUIRED_ARTIFACTS if not os.path.exists(f)]
assert not missing, (
    f"⛔ Artefak tidak ditemukan: {missing}\n"
    "Jalankan notebook sebelumnya terlebih dahulu."
)
print("✅ Semua artefak yang dibutuhkan tersedia.")

✅ Semua artefak yang dibutuhkan tersedia.


In [4]:
# ── RULE-TRAIN-02: Muat Artefak dari Preprocessing ─────────────────────
import json

train_data  = np.load(f"{ARTIFACTS_DIR}/data_train.npz")
X_train     = train_data["X_train"]
y_train     = train_data["y_train_cls"]
y_train_reg = train_data["y_train_reg"]

with open(f"{ARTIFACTS_DIR}/label_info.json") as f:
    label_info = json.load(f)

LOOK_BACK  = label_info["look_back"]
N_FEATURES = label_info["n_features"]

print(f"[LOAD] X_train : {X_train.shape}")
print(f"[LOAD] y_train : {y_train.shape}  (0={( y_train==0).sum()}, 1={(y_train==1).sum()})")

[LOAD] X_train : (3027, 24, 3)
[LOAD] y_train : (3027,)  (0=1163, 1=1864)


In [5]:
# ── RULE-TRAIN-03: Penanganan Class Imbalance ──────────────────────────
from sklearn.utils.class_weight import compute_class_weight

# Gunakan data asli tanpa oversampling manual
classes  = np.unique(y_train)
cw_arr   = compute_class_weight("balanced", classes=classes, y=y_train)
cw_dict  = dict(zip(classes.tolist(), cw_arr.tolist()))

print(f"[CW] Class weights yang diterapkan: {cw_dict}")

[CW] Class weights yang diterapkan: {0: 1.3013757523645744, 1: 0.8119635193133047}


In [6]:
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Lambda, Add, Activation
from tensorflow.keras import Model

def build_model(look_back: int, n_features: int) -> Model:
    inputs = Input(shape=(look_back, n_features))
    x = LSTM(64, return_sequences=True)(inputs)
    x = Dropout(0.2)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(0.2)(x)

    # Sensor dibaca tiap 2 menit, sehingga nilai soil terakhir adalah baseline terkuat.
    # LSTM belajar koreksi residual kecil, bukan memprediksi ulang dari nol.
    last_soil = Lambda(lambda t: t[:, -1, 2:3], output_shape=(1,), name="last_soil")(inputs)

    reg_x = Dense(16, activation="relu")(x)
    delta = Dense(1, activation="tanh", name="soil_delta_raw")(reg_x)
    delta = Lambda(lambda z: z * 0.15, output_shape=(1,), name="soil_delta_scaled")(delta)
    reg_added = Add(name="soil_residual_add")([last_soil, delta])
    output_reg = Activation("linear", name="regression")(reg_added)

    cls_x = Dense(16, activation="relu")(x)
    output_cls = Dense(1, activation="sigmoid", name="classification")(cls_x)

    model = Model(inputs=inputs, outputs=[output_cls, output_reg], name="bonsai_lstm_residual")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss={"classification": "binary_crossentropy", "regression": "mse"},
        metrics={
            "classification": ["accuracy", tf.keras.metrics.AUC(name="auc")],
            "regression": ["mae"],
        },
        loss_weights={"classification": 0.5, "regression": 50.0},
    )
    return model


In [7]:
model = build_model(LOOK_BACK, N_FEATURES)
model.summary()

Model: "bonsai_lstm_residual"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 24, 3)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 24, 64)    │     17,408 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 24, 64)    │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     12,416 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        528 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ soil_delta_raw      │ (None, 1)         │         17 │ dense[0][0]       │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ last_soil (Lambda)  │ (None, 1)         │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ soil_delta_scaled   │ (None, 1)         │          0 │ soil_delta_raw[0… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        528 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ soil_residual_add   │ (None, 1)         │          0 │ last_soil[0][0],  │
│ (Add)               │                   │            │ soil_delta_scale… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classification      │ (None, 1)         │         17 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regression          │ (None, 1)         │          0 │ soil_residual_ad… │
│ (Activation)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 30,914 (120.76 KB)

 Trainable params: 30,914 (120.76 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# ── RULE-TRAIN-05: Callbacks Wajib ──────────────────────────────────────
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    CSVLogger(HISTORY_PATH, separator=",", append=False),
]
print("✅ Callbacks configured.")

✅ Callbacks configured.


In [9]:
# ── RULE-TRAIN-06: Eksekusi Pelatihan ──────────────────────────────────
# Normalisasi target regresi ke range 0-1 (bagi 100)
y_train_reg_scaled = (y_train_reg / 100.0).astype("float32")
y_train_cls_fix = y_train.reshape(-1, 1).astype("float32")
y_train_reg_fix = y_train_reg_scaled.reshape(-1, 1).astype("float32")

# Create sample weights for each head
sample_weights_cls = np.array([cw_dict[int(y)] for y in y_train], dtype="float32")
sample_weights_reg = np.ones_like(y_train, dtype="float32")

# Keras 3 expects sample_weight to have same structure as output labels (list)
sample_weight_list = [sample_weights_cls, sample_weights_reg]

history = model.fit(
    X_train, [y_train_cls_fix, y_train_reg_fix],
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    validation_split = VAL_SPLIT,
    callbacks        = callbacks,
    sample_weight    = sample_weight_list, # type: ignore
    verbose          = 2,
)
print(f"[TRAIN] Model disimpan -> {MODEL_PATH}")
print(f"[TRAIN] Histori disimpan -> {HISTORY_PATH}")

Epoch 1/100



Epoch 1: val_loss improved from None to 0.87431, saving model to ../artifacts/model_bonsai_lstm.keras



Epoch 1: finished saving model to ../artifacts/model_bonsai_lstm.keras


76/76 - 5s - 60ms/step - classification_accuracy: 0.6770 - classification_auc: 0.4360 - classification_loss: 0.6315 - loss: 0.4410 - regression_loss: 0.0025 - regression_mae: 0.0281 - val_classification_accuracy: 0.0165 - val_classification_auc: 0.7184 - val_classification_loss: 1.1010 - val_loss: 0.8743 - val_regression_loss: 0.0065 - val_regression_mae: 0.0643


Epoch 2/100


...


Epoch 39: val_loss did not improve from 0.39608


76/76 - 1s - 11ms/step - classification_accuracy: 0.8905 - classification_auc: 0.9118 - classification_loss: 0.3099 - loss: 0.2790 - regression_loss: 0.0025 - regression_mae: 0.0283 - val_classification_accuracy: 0.9835 - val_classification_auc: 0.9708 - val_classification_loss: 0.1766 - val_loss: 0.4128 - val_regression_loss: 0.0065 - val_regression_mae: 0.0647


Epoch 39: early stopping


Restoring model weights from the end of the best epoch: 14.


[TRAIN] Model disimpan -> ../artifacts/model_bonsai_lstm.keras
[TRAIN] Histori disimpan -> ../artifacts/training_history.csv


In [10]:
# ── RULE-TRAIN-07: Tampilkan Ringkasan Training ────────────────────────
hist_df    = pd.read_csv(HISTORY_PATH)
best_epoch = int(hist_df["val_loss"].idxmin()) + 1

print(f"\n[SUMMARY] Epoch terbaik    : {best_epoch} / {len(hist_df)}")
print(f"[SUMMARY] Val Loss terbaik : {hist_df['val_loss'].min():.4f}")
print(f"[SUMMARY] Val Accuracy     : {hist_df['val_classification_accuracy'].iloc[best_epoch-1]:.4f}")
print(f"[SUMMARY] Val AUC          : {hist_df['val_classification_auc'].iloc[best_epoch-1]:.4f}")


[SUMMARY] Epoch terbaik    : 14 / 39
[SUMMARY] Val Loss terbaik : 0.3961
[SUMMARY] Val Accuracy     : 0.9835
[SUMMARY] Val AUC          : 0.9749


## 📊 Ringkasan Training

**Model yang dihasilkan:**
- `model_bonsai_lstm.keras` → Model LSTM terbaik (berdasarkan val_loss minimum)
- `training_history.csv` → Histori loss, accuracy, AUC per epoch

**Langkah selanjutnya:**
Jalankan `notebooks/03_testing.ipynb` untuk menjalankan prediksi pada data testing.